In [1]:
import json


In [ ]:
import openai

# Set your OpenAI API key
openai.api_key = "key"

# System Prompt

# Function to generate commonsense hypothesis

In [3]:
def generate_commonsense_hypothesis(premise):
    system_prompt = """
        You are a commonsense reasoning and moral philosophy expert. Your task involves, generating a "hypothesis" statement base on
        common sense factors related to the suituation being evaluated.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Premise: {premise}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
    )
    return response.choices[0].message.content

In [4]:
test_premise = "Today would have been my best friend's 18th birthday, we'd be going out together for the first time, we'd be sitting here making resolutions for the new year that we both know we'd never keep. None of that is even possible though, because he's gone. It was never supposed to be like this."
hypo = generate_commonsense_hypothesis(test_premise)
print(hypo)

Hypothesis: The loss of a close friend at a young age can deeply impact one's sense of loss, creating a void in shared experiences and future possibilities, leading to a profound sense of sadness and longing for what could have been.


# Answer generator

In [5]:
def generate_answer(premise):
    system_prompt = """
        Your task is to first categorize the mental state of the person who wrote the premise into one of following 6 categories;
        C1: No Reason
        C2: Jobs and careers related stressed
        C3: Medication related stressed
        C4: Relationship related stressed
        C5: Alienation related stressed
        C6: Bias or abuse related stressed
        And generate a reasoning for the selected category.

        Output format should be python dict object with the following structure:
        ```{"Category": "<selected category>", "Reasoning": "<reasoning>"}```

        Example :
        Input :
            premise: I'm anxious because of a deadline.
        Output :
        {"Category": "C2",
        "Reasoning": "Deadlines often cause stress in jobs and careers."}
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Premise: {premise}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages
    )
    return response.choices[0].message.content

In [6]:
test_premise = "Today would have been my best friend's 18th birthday, we'd be going out together for the first time, we'd be sitting here making resolutions for the new year that we both know we'd never keep. None of that is even possible though, because he's gone. It was never supposed to be like this."
output = generate_answer(test_premise)
print(output)

{"Category": "C4",
"Reasoning": "The person is experiencing relationship-related stress as they are grieving the loss of their best friend and feeling the absence of their presence on what would have been a special occasion. The sense of loss and longing for their friend's presence indicates the impact of the relationship on their mental state."}


In [7]:
out = json.loads(output)
out

{'Category': 'C4',
 'Reasoning': "The person is experiencing relationship-related stress as they are grieving the loss of their best friend and feeling the absence of their presence on what would have been a special occasion. The sense of loss and longing for their friend's presence indicates the impact of the relationship on their mental state."}

# Function to perform ReAct iterations

In [8]:
def react_iteration(premise, hypothesis):
    system_prompt = """
        You are a commonsense reasoning and moral philosophy expert. Your tasks is to do natual language inference between premise and hypothesis.
        So, you need to generate an **action** out of following options;
            entailment: one which is necessarily true or appropriate whenever the premise is true;
            contradiction: one which is necessarily false or inappropriate whenever the premise is true;
            neutral: and one where neither condition applies.
        just output a single word option.
    """
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Premise: {premise}\nHypothesis: {hypothesis}"}
    ]
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        max_tokens=150,
        temperature=0.5
    )
    return response.choices[0].message.content

In [9]:
reason = "The emotional tone of the premise suggests deep sorrow and longing due to the loss of a best friend, indicating a significant impact on relationships. The individual is grieving the absence of their friend on what should have been a joyous occasion, leading to relationship-related stress."
hypo = "The premature loss of a close friend, especially during a significant milestone like their 18th birthday, can lead to profound feelings of grief, regret, and the haunting sense of unfulfilled potential and missed opportunities."
resp = react_iteration(reason, hypo)
print(resp)

entailment


# Function to finalize the category

In [10]:
def inference(statement, max_iter=10):
    hypothesis = generate_commonsense_hypothesis(statement)
    reasoning = generate_answer(statement)
    cat, reason = json.loads(reasoning).values()
    action = react_iteration(reason, hypothesis)

    i = 0
    while i < max_iter:
        if action.lower() == "entailment":
            return cat, reason
        else:
            if action.lower() == "neutral" and i==max_iter:
                return cat, reason
            else:
                cat, reason = json.load(reasoning).values()
                i += 1

    return cat, reason

In [12]:
import pandas as pd
import tqdm
import sys

batch_size = 1000

df = pd.read_csv("CAMS.csv")

all_rows = list(df.iterrows())
all_rows = all_rows[4606:]

for batch_idx in range((len(df)//batch_size) + 1):
    print("Processing batch", batch_idx)
    results = []
    data = all_rows[batch_size*batch_idx: batch_size*(batch_idx + 1)]
    for i, row in tqdm.tqdm(data):
      try:
        cat, reason = inference(row["text"])
        results.append({
            "index": i,
            "text": row["text"],
            "category": cat,
            "reason": reason,
            "orig_category": row["category"],
            "orig_reason": row["explanation"],
        })
      except KeyboardInterrupt:
          print("KeyboardInterrupt received, exiting gracefully...")
          sys.exit(0)
      except Exception as e:
          print(f"Failed processing batch {batch_idx}-{i} due to error: {e}")

    res_df = pd.DataFrame(results)
    res_df.to_csv(f"out-{batch_size}-{batch_idx}.csv")


Processing batch 0


  3%|▎         | 14/446 [00:30<13:24,  1.86s/it]

Failed processing batch 0-4619 due to error: 'str' object has no attribute 'read'


  6%|▌         | 27/446 [00:57<13:46,  1.97s/it]

Failed processing batch 0-4632 due to error: 'str' object has no attribute 'read'


...

0it [00:00, ?it/s]


Processing batch 4


0it [00:00, ?it/s]


Processing batch 5


0it [00:00, ?it/s]


In [ ]:

  res_df = pd.DataFrame(results)
  res_df.to_csv(f"out-{batch_size}-{batch_idx}tmp.csv")